In [1]:
# importing Database

import os

from dotenv import load_dotenv

from langchain_community.utilities import SQLDatabase

load_dotenv()

mysql_uri = (
    f"mysql+pymysql://"
    f"{os.getenv('MYSQL_USER')}:"
    f"{os.getenv('MYSQL_PASSWORD')}@"
    f"{os.getenv('MYSQL_HOST')}:"
    f"{os.getenv('MYSQL_PORT')}/"
    f"{os.getenv('MYSQL_DATABASE')}"
)

db = SQLDatabase.from_uri(mysql_uri)

print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())

C:\Users\jassi\AppData\Local\Temp\ipykernel_22952\1424904557.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Dialect: mysql
Tables: ['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'sales_orders_old', 'state_regions']


In [2]:
# importing llm

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:4b",
    temperature=0,
)

In [3]:
# importing SQL Tools

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db = db,
    llm = llm
)

tools = toolkit.get_tools()

In [4]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print()

sql_db_query
Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema
Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables
Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker
Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [5]:
system_prompt = f"""
You are an agent designed to interact with a MySQL database.

Given a user's question, determine the relevant tables and columns,
generate a syntactically correct MySQL SQL query, execute it, and
return a clear natural-language answer.

Rules:

1. Always inspect the available tables first.
2. Retrieve the schema of relevant tables before generating SQL.
3. Only query tables that are relevant to the user's question.
4. Only select columns that are necessary.
5. Always check the SQL query before executing it.
6. If the SQL query produces an error, analyze the error and fix the query.
7. NEVER execute INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE,
   CREATE, or other data-modifying statements.
8. Only perform read-only queries.
9. Do not expose unnecessary database details to the user.
10. Return the final answer in clear natural language.

11. After the SQL query has been checked and approved,
    ALWAYS execute the query using sql_db_query before producing
    the final answer.

12. NEVER infer numerical results from sample rows returned by
    sql_db_schema. Numerical answers must come from executing SQL.

13. Do not answer a database question using information from the
    schema sample rows when the answer can be obtained by executing SQL.

Database dialect: {db.dialect}
"""

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model = llm,
    tools=tools,
    system_prompt=system_prompt
)

In [7]:
question = "What was the total education budget in 2017?"

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

KeyboardInterrupt: 

In [ ]:
print(result["messages"][-1].content)

[{'type': 'text', 'text': "I could not find a specific education budget in the database for 2017. The available budget data is categorized by 'Product Name', and 'Education' is not listed as a distinct product.", 'extras': {'signature': 'CuYQARFNMg8lkCctzsXagP0p9TMEbJUj3f2QW7gKCkTLhvKfEPJghfU2+cnCWxbu4z0SxQMqVWNhCRak8QUe+xUt12e8dzs5IPZOw7qiFGp1gOsdSW3e4yKgHDrXE0Z4B38Dw9XDEIRmBzXAKRzlO2LYVCo87UXxsLhHEhaEmM8PWO2kMZ6Wzxt1AHkV++V8QO3ogKd8sCgSYEQbLgabe7EFU0bRiCnGIGYAehqIl2cUHWZGS/YcGvPt5KUmld08BgOevtmS8IXlcQHjrnOpupIN3YdXzguf7PxPHSTN1i8SqrO6g9AFhyPguQpG3p1zxDIR2RE2TPKJknAhbWxM39siRCuQ3hfkyTN6PFYrZMV7P7jQmnCR4CQSadK1ejm1ilAlTuYkPhnkCjULOrmk49yJIwYKvbs98wXhe7+iV+MjPlCTM4KrTEU11kEToOv2Akvatubt/irYpOrXxsCk85TG/3/M7REAlQt/FCtd7XahwAy/39iKQjNJFLppC/oj4aCujbt4mHgRkUDewAPj75qzzD3n++gUDPQWm0edLcFH0sBQczz5KxsjEpybnvx/lOT97DEX2reYQauM4GUSJoUujplPxfQGV0z63k7b2+fMz0xPrkS+X+IlHHnyrlqgRLm+4cWdFED4grzZaQZjBiY6Xl3N21GCCBiH94Ec7CuLbKwM4sidS7LXU/Lo50QOSMggjVcCZGR9qUHJaeU7sMnxBpDT4c8OQRl4bpK+dzLPVmgfd2YhcQ4OE+

In [9]:
for event in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which state generated the highest total sales amount?"
            }
        ]
    },
    stream_mode="updates",
):
    for node, data in event.items():

        print(f"\n===== {node} =====")

        if "messages" in data:
            for message in data["messages"]:
                print(message)


===== model =====
content='' additional_kwargs={} response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-08T11:44:02.0311835Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24655699800, 'load_duration': 2923400, 'prompt_eval_count': 741, 'prompt_eval_duration': 215336000, 'eval_count': 529, 'eval_duration': 24162264000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'} id='lc_run--01a080d4-af9d-7373-9bc5-272f4c15b70b-0' tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'fd2e3131-cbaa-42fd-ba9c-f317b5ef7379', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 741, 'output_tokens': 529, 'total_tokens': 1270}

===== tools =====
content='2017_budgets, customers, products, regions, sales_order, sales_orders_old, state_regions' name='sql_db_list_tables' id='4a1ea1dd-ff1e-4031-8232-219a91f59d5b' tool_call_id='fd2e3131-cbaa-42fd-ba9c-f317b5ef7379'

===== model =====
content='' additional_kwargs={} resp